# Notebook Replica Set (INF-239)

Registro de las ejecuciones realizadas para la Tarea 3 de INF-239. Cada seccion corresponde a un requisito y muestra la evidencia capturada.

### Preparacion rapida del notebook

- Se importan las librerias basicas.
- Se definen las rutas al dataset y al JSON exportado.
- Se declara `run_cmd` para repetir los comandos tal como se corrieron en la terminal.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import subprocess

PROYECTO = Path('.').resolve()
DIRECTORIO_DISCURSOS = PROYECTO / 'DiscursosOriginales'
EXPORTACION_JSON = PROYECTO / 'discursos-final-202373011-4-202304074-6.json'
CADENA_CONEXION = 'mongodb://mongo1:30001,mongo2:30002,mongo3:30003/?replicaSet=my-replica-set&readPreference=primary&appname=MongoDB%20Compass&ssl=false'

print(f'Proyecto: {PROYECTO}')
print(f'Dataset: {DIRECTORIO_DISCURSOS} | Archivos: {len(list(DIRECTORIO_DISCURSOS.glob('*.txt')))}')
print(f'Ultima actualizacion del JSON final: {datetime.fromtimestamp(EXPORTACION_JSON.stat().st_mtime)}')

def ejecutar_comando(cmd, check=False):
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    if check and proc.returncode != 0:
        raise RuntimeError(proc.stderr)
    return proc

Proyecto: C:\Users\sofia\OneDrive\Escritorio\BDD\Tarea 3\Tarea 3\tarea3-bd
Dataset: C:\Users\sofia\OneDrive\Escritorio\BDD\Tarea 3\Tarea 3\tarea3-bd\DiscursosOriginales | Archivos: 680
Ultima actualizacion del JSON final: 2025-11-14 01:02:51.598394


### Requisito 1: Replica Set en Docker (3 nodos)

Se despliega la arquitectura con `docker compose -f deploy-202373011-4-202304074-6.yml up -d`, dejando registro de la ejecucion y de cualquier mensaje que entrega Docker.

In [2]:
# 1) Despliegue de los contenedores (requiere Docker Desktop corriendo)
resultado = ejecutar_comando(['docker', 'compose', '-f', 'deploy-202373011-4-202304074-6.yml', 'up', '-d'])
if resultado.returncode != 0:
    print('[WARN] Inicia Docker Desktop y vuelve a ejecutar la celda.')

$ docker compose -f deploy-202373011-4-202304074-6.yml up -d


time="2025-11-14T01:16:26-03:00" level=warning msg="C:\\Users\\sofia\\OneDrive\\Escritorio\\BDD\\Tarea 3\\Tarea 3\\tarea3-bd\\deploy-202373011-4-202304074-6.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container mongo1  Running
 Container mongo3  Running
 Container mongo2  Running
 Container rs-init  Running
 Container mongo1  Waiting
 Container mongo1  Waiting
 Container mongo1  Healthy
 Container mongo1  Healthy
 Container mongo3  Waiting
 Container mongo1  Waiting
 Container mongo2  Waiting
 Container mongo3  Healthy
 Container mongo2  Healthy
 Container mongo1  Healthy


In [3]:
# 2) Verificar el estado del Replica Set
from pymongo import MongoClient
try:
    cliente = MongoClient(CADENA_CONEXION, serverSelectionTimeoutMS=2000)
    estado = cliente.admin.command('hello')
    hosts = estado.get('hosts', [])
    if hosts and isinstance(hosts[0], str):
        miembros = hosts
    else:
        miembros = [host.get('name', str(host)) for host in hosts]
    print(f"Replica Set: {estado.get('setName')} | Primario: {estado.get('primary')}")
    print('Miembros:', miembros)
except Exception as exc:
    print('No fue posible consultar el Replica Set. Mensaje recibido:')
    print(exc)
    print('Ejecuta la celda nuevamente cuando los contenedores esten activos.')

Replica Set: my-replica-set | Primario: mongo1:30001
Miembros: ['mongo1:30001', 'mongo2:30002', 'mongo3:30003']


### Requisito 2: Preprocesamiento e ingesta del corpus

- **Script usado:** `conectarMongo.py`.
- `python conectarMongo.py --ingest --skip-db` genera el JSON final (`discursos-final-202373011-4-202304074-6.json`) y confirma que existen 680 discursos.
- `python conectarMongo.py --ingest` (sin `--skip-db`) inserta los documentos en `Politica.Discursos` utilizando el Replica Set.
- El notebook muestra los mensajes completos de ambas ejecuciones para respaldar el proceso.

In [4]:
# 3) Ingesta del corpus (quitar --skip-db cuando Mongo este arriba)
ejecutar_comando(['python', 'conectarMongo.py', '--ingest', '--skip-db'])

$ python conectarMongo.py --ingest --skip-db


Archivo JSON generado en discursos-final-202373011-4-202304074-6.json
Batches: 100%|##########| 43/43 [00:16<00:00,  2.61it/s]


CompletedProcess(args=['python', 'conectarMongo.py', '--ingest', '--skip-db'], returncode=0, stdout='Archivo JSON generado en discursos-final-202373011-4-202304074-6.json\n', stderr='\nBatches:   0%|          | 0/43 [00:00<?, ?it/s]\nBatches:   2%|2         | 1/43 [00:00<00:22,  1.89it/s]\nBatches:   5%|4         | 2/43 [00:00<00:19,  2.07it/s]\nBatches:   7%|6         | 3/43 [00:01<00:18,  2.16it/s]\nBatches:   9%|9         | 4/43 [00:01<00:17,  2.23it/s]\nBatches:  12%|#1        | 5/43 [00:02<00:16,  2.30it/s]\nBatches:  14%|#3        | 6/43 [00:02<00:15,  2.34it/s]\nBatches:  16%|#6        | 7/43 [00:03<00:15,  2.39it/s]\nBatches:  19%|#8        | 8/43 [00:03<00:14,  2.43it/s]\nBatches:  21%|##        | 9/43 [00:03<00:14,  2.40it/s]\nBatches:  23%|##3       | 10/43 [00:04<00:13,  2.43it/s]\nBatches:  26%|##5       | 11/43 [00:04<00:13,  2.46it/s]\nBatches:  28%|##7       | 12/43 [00:05<00:12,  2.46it/s]\nBatches:  30%|###       | 13/43 [00:05<00:11,  2.51it/s]\nBatches:  33%|###2   

In [5]:
# 4) Inspeccion del JSON final generado
with EXPORTACION_JSON.open('r', encoding='utf-8') as archivo:
    documentos = json.load(archivo)

print(f'Se procesaron {len(documentos)} discursos.')
print('Ejemplo de documento:')
ejemplo = documentos[0]
print({'_id': ejemplo['_id'], 'archivo': ejemplo['metadata']['filename']})
print('Primeros 180 caracteres del texto original:')
print(ejemplo['texto'][:180].replace('\n', ' ') + '...')

Se procesaron 680 discursos.
Ejemplo de documento:
{'_id': '44b5a3c1bb8f59b2593341223422c48ed4f78fb31ff70a0781751434b44419cd', 'archivo': '100020.txt'}
Primeros 180 caracteres del texto original:
Muy buenos días:     Disfruten de este Sol de invierno. Gabriela Mistral decía que “hay que amar a nuestras ciudades” y agregaba que la belleza de nuestras ciudades nos ennoblece y...


### Requisito 3: Consulta semantica que devuelve top 5

El mismo script `conectarMongo.py` implementa la consulta pedida. Con el JSON cargado se ejecuta `conectarMongo.py --query "Educacion publica y crecimiento economico"`. El notebook ilustra como el modelo `all-MiniLM-L6-v2` calcula el embedding de la consulta, obtiene la similitud coseno y lista los cinco discursos con mayor puntaje.

In [6]:
# 5) Consulta semantica offline (usa el JSON exportado para mostrar resultados)
import numpy as np
from sentence_transformers import SentenceTransformer

consulta = 'Educacion publica y crecimiento economico'
modelo = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embedding_consulta = modelo.encode([consulta], convert_to_numpy=True, normalize_embeddings=True)[0]
matriz_embeddings = np.array([doc['embedding'] for doc in documentos], dtype=np.float32)
puntajes = matriz_embeddings @ embedding_consulta
indices = np.argsort(-puntajes)[:5]
for posicion, indice in enumerate(indices, start=1):
    doc = documentos[indice]
    fragmento = doc['texto'][:160].replace('\n', ' ')
    print(f"[{posicion}] {doc['metadata']['filename']} | score={puntajes[indice]:.4f}")
    print(fragmento + '...')
    print('-' * 70)

C:\Users\sofia\AppData\Local\Programs\Python\Python313\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


[1] 82157.txt | score=0.6314
Muy buenos días:     Quiero saludar con mucho cariño a la ministra de Educación, al presidente de la FIDE, a los directivos, pero principalmente a todos ustedes...
----------------------------------------------------------------------
[2] 80522.txt | score=0.5985
Muy buenos días:     La verdad es que aquí se ve el espíritu de innovación, de emprendimiento, cómo ha cambiado este barrio, cómo está mejorando la calidad de n...
----------------------------------------------------------------------
[3] 97967.txt | score=0.5660
Muchas gracias Primer Ministro Abe, por esta cálida bienvenida.     Tanto la teoría económica como la historia económica nos demuestran en forma elocuente que h...
----------------------------------------------------------------------
[4] 86296.txt | score=0.5638
Muy buenos días:     Hoy hemos recibido, de parte de este grupo de trabajo, presidido por la ex ministra Mariana Aylwin, este Informe “Todos al Aula”, que es un...
---------------

### Requisito 4: Robustez del sistema

- **Consistencia**: se ejecuta `countDocuments()` en cada nodo para comprobar que el numero de registros coincide.
- **Alta disponibilidad**: se detiene `mongo1` mediante `docker stop` y se evidencia el cambio de rol antes de volver a iniciarlo.
- Toda la salida se deja tal cual aparece para respaldar las pruebas.

In [7]:
# 6) Consistencia: conteo de documentos replicados en los tres nodos
comandos = [
    ['docker', 'exec', 'mongo1', 'mongosh', '--port', '30001', '--quiet', '--eval', "db.getSiblingDB('Politica').Discursos.countDocuments()"],
    ['docker', 'exec', 'mongo2', 'mongosh', '--port', '30002', '--quiet', '--eval', "db.getSiblingDB('Politica').Discursos.countDocuments()"],
    ['docker', 'exec', 'mongo3', 'mongosh', '--port', '30003', '--quiet', '--eval', "db.getSiblingDB('Politica').Discursos.countDocuments()"]
]
for comando in comandos:
    ejecutar_comando(comando, check=True)

$ docker exec mongo1 mongosh --port 30001 --quiet --eval db.getSiblingDB('Politica').Discursos.countDocuments()


679
$ docker exec mongo2 mongosh --port 30002 --quiet --eval db.getSiblingDB('Politica').Discursos.countDocuments()


679
$ docker exec mongo3 mongosh --port 30003 --quiet --eval db.getSiblingDB('Politica').Discursos.countDocuments()


679


In [8]:
# 7) Alta disponibilidad: detener el primario y recuperar el cluster
import time

ejecutar_comando(['docker', 'stop', 'mongo1'], check=True)
ejecutar_comando(['docker', 'exec', 'mongo2', 'mongosh', '--port', '30002', '--quiet', '--eval', "rs.status().members.map(m => ({name: m.name, state: m.stateStr}))"], check=True)
ejecutar_comando(['python', 'conectarMongo.py', '--query', 'Mensaje presidencial sobre educacion publica'], check=True)
ejecutar_comando(['docker', 'start', 'mongo1'], check=True)
time.sleep(8)
ejecutar_comando(['docker', 'exec', 'mongo2', 'mongosh', '--port', '30002', '--quiet', '--eval', "rs.status().members.map(m => ({name: m.name, state: m.stateStr}))"], check=True)

$ docker stop mongo1


mongo1
$ docker exec mongo2 mongosh --port 30002 --quiet --eval rs.status().members.map(m => ({name: m.name, state: m.stateStr}))


[
  { name: 'mongo1:30001', state: '(not reachable/healthy)' },
  { name: 'mongo2:30002', state: 'PRIMARY' },
  { name: 'mongo3:30003', state: 'SECONDARY' }
]
$ python conectarMongo.py --query Mensaje presidencial sobre educacion publica


Top 5 resultados para la consulta: "Mensaje presidencial sobre educacion publica"

[1] ID: a6458b4fa6d85ad7d370dc28cf0f9425d5b4f4724447d39ab95a781bd5145801 | archivo: 82157.txt
   Puntaje coseno: 0.5579
   Fragmento: Muy buenos días:     Quiero saludar con mucho cariño a la ministra de Educación, al presidente de la FIDE, a los directivos, pero principalmente a todos ustedes, que son los que durante muchas décadas le han dado vida, a...
--------------------------------------------------------------------------------
[2] ID: 14fb5c3abf3adc8011c1dd76f557a310f18f64457d12880d072b3eee3d9ef093 | archivo: 77982.txt
   Puntaje coseno: 0.4973
   Fragmento: Aquí estamos con nuestras nuevas profesoras, Dairis e Isidora, que nos han enseñado a programar. Y la verdad es que es muy importante lo que están haciendo con este programa, porque lo que viene en el futuro es un mundo ...
--------------------------------------------------------------------------------
[3] ID: 687835a27f2920a9753cf208b3578fe

mongo1


$ docker exec mongo2 mongosh --port 30002 --quiet --eval rs.status().members.map(m => ({name: m.name, state: m.stateStr}))


[
  { name: 'mongo1:30001', state: '(not reachable/healthy)' },
  { name: 'mongo2:30002', state: 'PRIMARY' },
  { name: 'mongo3:30003', state: 'SECONDARY' }
]


CompletedProcess(args=['docker', 'exec', 'mongo2', 'mongosh', '--port', '30002', '--quiet', '--eval', 'rs.status().members.map(m => ({name: m.name, state: m.stateStr}))'], returncode=0, stdout="[\n  { name: 'mongo1:30001', state: '(not reachable/healthy)' },\n  { name: 'mongo2:30002', state: 'PRIMARY' },\n  { name: 'mongo3:30003', state: 'SECONDARY' }\n]\n", stderr='')

In [9]:
# 6) Limpieza opcional de la infraestructura
ejecutar_comando(['docker', 'compose', '-f', 'deploy-202373011-4-202304074-6.yml', 'down'])

$ docker compose -f deploy-202373011-4-202304074-6.yml down


time="2025-11-14T01:17:33-03:00" level=warning msg="C:\\Users\\sofia\\OneDrive\\Escritorio\\BDD\\Tarea 3\\Tarea 3\\tarea3-bd\\deploy-202373011-4-202304074-6.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container rs-init  Stopping
 Container rs-init  Stopped
 Container rs-init  Removing
 Container rs-init  Removed
 Container mongo2  Stopping
 Container mongo3  Stopping
 Container mongo3  Stopped
 Container mongo3  Removing
 Container mongo3  Removed
 Container mongo2  Stopped
 Container mongo2  Removing
 Container mongo2  Removed
 Container mongo1  Stopping
 Container mongo1  Stopped
 Container mongo1  Removing
 Container mongo1  Removed
 Network tarea3-bd_mongo-net  Removing
 Network tarea3-bd_mongo-net  Removed


CompletedProcess(args=['docker', 'compose', '-f', 'deploy-202373011-4-202304074-6.yml', 'down'], returncode=0, stdout='', stderr='time="2025-11-14T01:17:33-03:00" level=warning msg="C:\\\\Users\\\\sofia\\\\OneDrive\\\\Escritorio\\\\BDD\\\\Tarea 3\\\\Tarea 3\\\\tarea3-bd\\\\deploy-202373011-4-202304074-6.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"\n Container rs-init  Stopping\n Container rs-init  Stopped\n Container rs-init  Removing\n Container rs-init  Removed\n Container mongo2  Stopping\n Container mongo3  Stopping\n Container mongo3  Stopped\n Container mongo3  Removing\n Container mongo3  Removed\n Container mongo2  Stopped\n Container mongo2  Removing\n Container mongo2  Removed\n Container mongo1  Stopping\n Container mongo1  Stopped\n Container mongo1  Removing\n Container mongo1  Removed\n Network tarea3-bd_mongo-net  Removing\n Network tarea3-bd_mongo-net  Removed\n')